[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/notebooks/profiling_and_tuning.ipynb)

# OpenImpala — Profiling and Performance Tuning

This notebook is the diagnostic complement to the seven user tutorials. Where the tutorials show *what to do*, this one shows *why a given solve takes the time it does* — and what to change when it takes too long.

The headline result, summarised up front: **the default solver `solver="auto"` is now MLMG** (AMReX's matrix-free geometric multigrid), the only option that scales near-linearly with voxel count. The notebook validates that experimentally on real porespy data, compares it against every HYPRE Krylov + multigrid configuration, and provides the diagnostics needed to spot regressions in future versions.

## Contents

| Section | Topic |
|---|---|
| 1   | Setup — environment detection and dependency install |
| 1a  | Build verification — confirm the GPU build is actually active |
| 2   | Synthetic datasets at 16³ – 128³ |
| **3** | **Solver bake-off on porous media — the headline diagnostic** |
| **4** | **Scaling validation (4 solvers) — does compute hold O(N)?** |
| 5   | `max_grid_size` tuning |
| 6   | NVIDIA Nsight Systems — GPU kernel timeline (optional) |
| 7   | Appendix — binding overhead diagnostics |
| 8   | Summary & HPC recommendations |

Sections 1 – 5 run on any environment in a few minutes. Section 6 produces the most useful output on a Colab GPU runtime (T4 / A100 / L4) with `nsys` available.


## 1. Setup

Detect hardware, install dependencies, import, start a session.

In [ ]:
import subprocess
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True, timeout=10)
    gpu_name = r.stdout.strip()
    print(f"GPU detected: {gpu_name}" if gpu_name else "No GPU — CPU only.")
except FileNotFoundError:
    gpu_name = ""
    print("nvidia-smi not found — CPU only.")


In [ ]:
# OpenImpala ships two PyPI wheels:
#   openimpala       — pure-Python + SciPy fallback (CPU only)
#   openimpala-cuda  — compiled C++ HYPRE/AMReX backend with CUDA kernels
#
# On a GPU runtime we install the CUDA wheel; otherwise we fall back to the
# pure-Python package.
!apt-get install -y libopenmpi-dev > /dev/null 2>&1
if gpu_name:
    print("GPU detected — installing openimpala-cuda")
    !pip install -q openimpala-cuda porespy py-spy
else:
    print("No GPU detected — installing openimpala (CPU)")
    !pip install -q openimpala porespy py-spy
print("Dependencies installed.")

In [ ]:
import openimpala as oi
from openimpala import core as oic  # noqa: F401
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, io, sys, re, cProfile, pstats, warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print(f"OpenImpala version: {oi.__version__}")


In [ ]:
session = oi.Session()
session.__enter__()
print("OpenImpala session started.")


## 1a. Build verification

Before profiling anything, confirm that the wheel `pip` installed actually has the GPU backend you expect. The most common cause of "OpenImpala feels slow on Colab" is a CPU-only wheel running on a GPU-equipped host — every solve in the rest of the notebook would be CPU-bound and ~10–50× slower than necessary on a single-phase 3D diffusion problem at 128³+.

`oi.build_info()` returns a dict of compile-time feature flags plus the runtime GPU device count. We check that the backend matches the hardware, and warn loudly if it doesn't.

In [ ]:
info = oi.build_info()
backend = info["backend"]   # one of: cpp-cuda, cpp-hip, cpp-cpu, pure-python

print(f"openimpala backend: {backend}")
print(f"  version:        {info.get('version', 'unknown')}")
print(f"  OpenMP threads: {info.get('openmp_max_threads', 1)}")
print(f"  MPI enabled:    {info.get('mpi_enabled', False)}")
print(f"  HYPRE CUDA:     {info.get('hypre_cuda', False)}")
print(f"  TinyProfile:    {info.get('tiny_profile', False)}")
if info.get("gpu_device_count", -1) > 0:
    print(f"  GPU devices:    {info['gpu_device_count']}")

is_gpu_build = backend in ("cpp-cuda", "cpp-hip")

if gpu_name and not is_gpu_build:
    print()
    print("=" * 70)
    print("  WARNING: CPU-only build detected on a GPU-equipped host")
    print("=" * 70)
    print(f"  Host GPU:          {gpu_name}")
    print(f"  openimpala build:  {backend}")
    print()
    print("  The GPU is idle. Every solve in this notebook will run on the CPU,")
    print("  which is typically 10–50× slower than the CUDA build for")
    print("  single-phase 3D diffusion on 128³+ grids.")
    print()
    print("  To fix:")
    print("    !pip uninstall -y openimpala")
    print("    !pip install openimpala-cuda")
    print("  Then Runtime → Restart session and re-run from the top.")
    print("=" * 70)
elif is_gpu_build:
    print(f"\n{backend.upper()} build active on {gpu_name or 'accelerator'}.")
else:
    print("\nCPU-only run (no accelerator detected).")

## 2. Synthetic datasets

PoreSpy is used to generate five synthetic porous structures at 16³, 32³, 64³, 96³, and 128³. Sizes are kept small enough that a full profiling run completes in minutes rather than hours, and large enough that the cubic scaling of voxel count is visible in the timings.

The 16³ dataset is small enough that its solve time is dominated by per-call setup rather than compute — useful for extracting the fixed overhead in §4.

In [ ]:
import porespy as ps

def make_micro(size, porosity=0.5, seed=42):
    np.random.seed(seed)
    im = ps.generators.blobs(shape=[size] * 3, porosity=porosity, blobiness=1.5)
    return im.astype(np.int32)

sizes = [16, 32, 64, 96, 128]
datasets = {n: make_micro(n) for n in sizes}
data_medium = datasets[64]  # default workhorse for most diagnostics

for n, d in datasets.items():
    print(f"  {n:>4d}^3  porosity={np.mean(d == 0):.3f}  bytes={d.nbytes/1e6:.2f} MB")


## 3. Solver bake-off on porous media

This is the headline diagnostic. All available HYPRE Krylov methods, all HYPRE multigrid preconditioners, and the matrix-free AMReX MLMG are run on the *same* porespy structure at 64³. Wall time and iteration count are reported side-by-side; solvers that fail (multigrid stalls on masked rows, etc.) are flagged in red.

**The default solver (`solver="auto"`, since v4.2.20) is MLMG** — it scales near-linearly (see §4) and has the lowest memory footprint of any option. The bake-off below confirms it's also the fastest on this geometry. The runner-up among HYPRE configurations is `flexgmres+pfmg`; `pcg+smg` is the safe fallback if MLMG ever stalls.

In [ ]:
import sys

# Sanity: make sure the porespy datasets cell (§2) has run. The restructure
# moved cells around; if you opened the notebook and jumped here directly,
# `data_medium` won't exist yet and the whole loop raises NameError before
# any print fires — looking exactly like "no output".
if "data_medium" not in dir():
    raise RuntimeError(
        "data_medium not defined — re-run §2 (Synthetic datasets) first."
    )

# Compare Krylov and multigrid choices on 64^3.
# Each entry is (label, solver, preconditioner). A preconditioner of None means
# the solver ignores it (standalone SMG/PFMG/Jacobi, or MLMG). `mlmg` is the
# AMReX-native matrix-free path and the library default since v4.3.0.
combos = [
    ("pcg",            "pcg",       None),
    ("pcg+pfmg",       "pcg",       "pfmg"),
    ("pcg+smg",        "pcg",       "smg"),
    ("flexgmres+pfmg", "flexgmres", "pfmg"),
    ("gmres",          "gmres",     None),
    ("gmres+pfmg",     "gmres",     "pfmg"),
    ("bicgstab",       "bicgstab",  None),
    ("smg",            "smg",       None),
    ("pfmg",           "pfmg",      None),
    ("mlmg",           "mlmg",      None),
]

print(f"Running bake-off on {data_medium.shape[0]}³ porespy data... "
      f"({len(combos)} solver/preconditioner combinations)", flush=True)

records = []
for label, s, pc in combos:
    kwargs = dict(phase=0, direction="z", solver=s, max_grid_size=32, verbose=0)
    if pc is not None:
        kwargs["preconditioner"] = pc

    # Flush before each call: a C++-level AMReX::Abort or CUDA OOM kills
    # the kernel without unwinding Python, so anything we haven't printed
    # already is lost. Live progress also makes it obvious *which* solver
    # killed the cell if one of them does.
    sys.stdout.flush()

    try:
        t0 = time.perf_counter()
        res = oi.tortuosity(data_medium, **kwargs)
        dt = time.perf_counter() - t0
        records.append({"label": label, "solver": s, "precond": pc, "t": dt,
                        "iters": res.iterations, "tau": res.tortuosity,
                        "ok": res.solver_converged})
        print(f"  {label:18s}  t={dt:6.2f}s  iters={res.iterations:4d}  "
              f"tau={res.tortuosity:.4f}", flush=True)
    except TypeError as e:
        if "preconditioner" in str(e) and pc is not None:
            print(f"  {label:18s}  SKIP — wheel predates preconditioner plumbing",
                  flush=True)
            records.append({"label": label, "solver": s, "precond": pc,
                            "t": np.nan, "iters": np.nan, "tau": np.nan, "ok": False})
            continue
        raise
    except Exception as e:
        records.append({"label": label, "solver": s, "precond": pc,
                        "t": np.nan, "iters": np.nan, "tau": np.nan, "ok": False})
        print(f"  {label:18s}  FAILED — {e}", flush=True)

df_solvers = pd.DataFrame(records)
df_solvers

In [ ]:
ok = df_solvers.dropna(subset=["t"])
if not ok.empty:
    best_row = ok.loc[ok["t"].idxmin()]
    best_label = best_row["label"]
    best_solver = best_row["solver"]
    best_precond = best_row["precond"]
else:
    best_label = "pcg"
    best_solver = "pcg"
    best_precond = None

fig, ax1 = plt.subplots(figsize=(11, max(4, 0.5 * len(df_solvers))))
colors = ["#2ecc71" if c else "#e74c3c" for c in df_solvers["ok"]]
y = np.arange(len(df_solvers))

ax1.barh(y, df_solvers["t"], color=colors, alpha=0.85, edgecolor="white")
ax1.set_yticks(y)
ax1.set_yticklabels(df_solvers["label"])
ax1.set_xlabel("Wall time (s)", color="#2c3e50")
ax1.set_title(f"Solver comparison — 64³, best by wall time: {best_label}",
              fontweight="bold")
ax1.invert_yaxis()

# Overlay iteration counts on a twin axis — this is the key diagnostic.
# A solver with few iterations but expensive each (multigrid) can still
# win overall; wall time alone hides that.
ax2 = ax1.twiny()
ax2.plot(df_solvers["iters"], y, "D", color="#8e44ad", markersize=9,
         markeredgecolor="white", markeredgewidth=1.2, zorder=5)
ax2.set_xlabel("Iterations", color="#8e44ad")
ax2.tick_params(axis="x", colors="#8e44ad")

# Annotate each row with iter count and per-iter cost
for i, (_, row) in enumerate(df_solvers.iterrows()):
    if row["ok"] and not np.isnan(row["t"]):
        per_iter = row["t"] / max(row["iters"], 1)
        ax1.text(row["t"] * 1.02, i,
                 f" {int(row['iters'])} iters  ({per_iter*1000:.0f} ms/iter)",
                 va="center", fontsize=8, color="#2c3e50")

from matplotlib.patches import Patch
legend = [
    Patch(facecolor="#2ecc71", label="Converged"),
    Patch(facecolor="#e74c3c", label="Failed"),
    plt.Line2D([0], [0], marker="D", color="#8e44ad", linestyle="None",
               markersize=8, label="Iterations"),
]
ax1.legend(handles=legend, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()

# Flag solvers with few iterations — likely multigrid/preconditioned.
# If one converged in <10 iterations but is slower per iteration, it may
# still be the better choice at larger problem sizes.
low_iter = ok[ok["iters"] < 10]
if not low_iter.empty:
    print("\nSolvers converging in <10 iterations (likely multigrid):")
    for _, r in low_iter.iterrows():
        print(f"  {r['label']:18s}  {int(r['iters'])} iters  {r['t']:.2f}s")
    print("  These scale O(N) and will overtake PCG at larger grids.")

print(f"\nBest solver at 64³ by wall time: {best_label} "
      f"(solver={best_solver}, preconditioner={best_precond})")


## 4. Scaling validation — does the solver hold O(N) compute?

Sweep four solvers across 32³ → 128³ on a uniform geometry and fit `t ~ N^p` log-log per solver. The line we want is near-linear (p ~ 1) — anything appreciably above 1.2 means iteration count is growing with problem size, which kills scalability to HPC-sized images.

Four solvers are compared:

* **`pcg`** — Krylov baseline with no preconditioning. The control.
* **`pcg+pfmg`** — fastest HYPRE at small N but its semicoarsening stalls on masked rows; included to show why it's not the default.
* **`flexgmres+pfmg`** — the bake-off's HYPRE winner. flexgmres handles PFMG's non-symmetric preconditioner application correctly.
* **`mlmg`** — the matrix-free AMReX path, the library default.

In [ ]:
scaling_sizes = [32, 64, 96, 128]
scaling_combos = [
    ("pcg",            {"solver": "pcg"}),
    ("pcg+pfmg",       {"solver": "pcg", "preconditioner": "pfmg"}),
    ("flexgmres+pfmg", {"solver": "flexgmres", "preconditioner": "pfmg"}),
    ("mlmg",           {"solver": "mlmg"}),
]

scaling_records = []
for label, kw in scaling_combos:
    for N in scaling_sizes:
        arr = np.zeros((N, N, N), dtype=np.int32)  # uniform phase 0 — always percolates
        try:
            t0 = time.perf_counter()
            res = oi.tortuosity(arr, phase=0, direction="z",
                                max_grid_size=min(32, N), verbose=0, **kw)
            dt = time.perf_counter() - t0
            scaling_records.append({"solver": label, "N": N, "t": dt,
                                    "iters": int(res.iterations),
                                    "ok": bool(res.solver_converged)})
            print(f"  {label:10s}  {N:3d}^3  t={dt:7.3f}s  iters={res.iterations:4d}")
        except TypeError as e:
            # Wheel predates the preconditioner kwarg — skip only those rows.
            msg = str(e)
            if ("preconditioner" in msg) and ("preconditioner" in kw):
                print(f"  {label:10s}  {N:3d}^3  SKIP — wheel predates preconditioner kwarg")
                scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                        "iters": 0, "ok": False})
                continue
            raise
        except Exception as e:
            scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                    "iters": 0, "ok": False})
            print(f"  {label:10s}  {N:3d}^3  FAILED — {e}")

df_scale = pd.DataFrame(scaling_records)
df_scale


In [ ]:
# Fit t = a · N^p in log-log per solver, then overlay the curves with an
# O(N) reference line and the O(N^1.76) baseline from §3b.
fig, ax = plt.subplots(figsize=(9, 5.5))
palette = {"pcg": "#e74c3c", "pcg+pfmg": "#2ecc71",
           "flexgmres+pfmg": "#3498db", "mlmg": "#8e44ad"}
fit_results = {}
for label, _ in scaling_combos:
    sub = df_scale[(df_scale["solver"] == label) & df_scale["ok"]].dropna(subset=["t"])
    if len(sub) < 2:
        fit_results[label] = np.nan
        continue
    p, loga = np.polyfit(np.log(sub["N"].astype(float)), np.log(sub["t"].astype(float)), 1)
    fit_results[label] = p
    ax.loglog(sub["N"], sub["t"], "o-", color=palette.get(label, "#333333"),
              markersize=8, linewidth=2, label=f"{label}  (p={p:.2f})")

# Reference lines — anchored at the first successful measurement
anchor = df_scale[df_scale["ok"]].dropna(subset=["t"]).sort_values("N")
if not anchor.empty:
    N_anchor = float(anchor.iloc[0]["N"])
    t_anchor = float(anchor.iloc[0]["t"])
    Ns = np.array(scaling_sizes, dtype=float)
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.0,
              "k--", alpha=0.4, linewidth=1, label="O(N) reference")
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.76,
              "r:", alpha=0.5, linewidth=1, label="O(N^1.76) (§3b baseline)")

ax.set_xlabel("Grid size N  (domain = N^3)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Scaling validation — multigrid vs. plain Krylov",
             fontweight="bold")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Verdict against the near-linear target (p < 1.2)
print("\nScaling exponents vs. near-linear target (p < 1.2):")
for label, p in fit_results.items():
    if np.isnan(p):
        print(f"  {label:10s}  n/a  (insufficient data)")
    elif p < 1.2:
        print(f"  {label:10s}  p = {p:.2f}  near-linear — target met")
    else:
        print(f"  {label:10s}  p = {p:.2f}  super-linear — target NOT met")

## 5. `max_grid_size` tuning

AMReX splits the domain into boxes of `max_grid_size` per side, one box per MPI rank (or GPU stream). Too small and the per-box overhead dominates; too large and load balancing suffers. The sweet spot is geometry- and hardware-dependent; this sweep finds it for the current platform.

In [ ]:
from openimpala import ConvergenceError

grid_sizes = [8, 16, 32, 64]
grid_rows = []
# Re-use the winner from §9 (best_solver / best_precond). On the porespy blob
# the winner is typically flexgmres+pfmg on GPU; on CPU PCG+SMG.
for gs in grid_sizes:
    kwargs = dict(phase=0, direction="z", solver=best_solver,
                  max_grid_size=gs, verbose=0)
    if best_precond is not None:
        kwargs["preconditioner"] = best_precond
    t0 = time.perf_counter()
    try:
        res = oi.tortuosity(data_medium, **kwargs)
        dt = time.perf_counter() - t0
        tau = res.tortuosity
        grid_rows.append({"max_grid_size": gs, "t": dt, "tau": tau, "ok": True})
        print(f"  max_grid_size={gs:3d}  t={dt:.2f}s  tau={tau:.4f}")
    except ConvergenceError as e:
        dt = time.perf_counter() - t0
        grid_rows.append({"max_grid_size": gs, "t": np.nan, "tau": np.nan, "ok": False})
        print(f"  max_grid_size={gs:3d}  FAILED ({e})")

df_grid = pd.DataFrame(grid_rows)
df_grid_ok = df_grid.dropna(subset=["t"])
if df_grid_ok.empty:
    best_gs = 32
    print("No successful grid sweeps — falling back to max_grid_size=32.")
else:
    best_gs = int(df_grid_ok.loc[df_grid_ok["t"].idxmin(), "max_grid_size"])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(df_grid_ok["max_grid_size"], df_grid_ok["t"], "o-", linewidth=2, markersize=9)
ax.set_xlabel("max_grid_size")
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Grid sweep — 64³, solver={best_label}", fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Optimal max_grid_size: {best_gs}")

## 6. NVIDIA Nsight Systems — GPU kernel timeline (optional)

If a GPU is available *and* `nsys` is installed, this section captures a kernel-level timeline: launches, memory transfers, idle gaps. Signatures of binding overhead leaking onto the GPU path are many tiny launches with gaps between them, or H2D transfers of the same data on every solve. On Colab `nsys` is not pre-installed; the cell below attempts an `apt-get` install and skips cleanly if that fails.

In [ ]:
import shutil

# Try to find nsys. On Colab it isn't pre-installed but can be apt-get'd
# in a few seconds. Pull it now if missing so §8 doesn't silently skip.
if shutil.which("nsys") is None and bool(gpu_name):
    print("nsys not on PATH — apt-get installing nsight-systems...")
    !apt-get install -y nsight-systems-cli > /dev/null 2>&1 || \
        apt-get install -y nsight-systems > /dev/null 2>&1
    if shutil.which("nsys") is None:
        # Fall back to whichever nsys ships under /opt/nvidia or similar
        import glob
        for p in glob.glob("/opt/nvidia/nsight-systems*/bin/nsys") + \
                glob.glob("/usr/local/cuda*/bin/nsys"):
            import os
            os.environ["PATH"] = os.path.dirname(p) + ":" + os.environ["PATH"]
            break

nsys_available = shutil.which("nsys") is not None and bool(gpu_name)

if not nsys_available:
    if not gpu_name:
        print("No GPU — skipping Nsight. (Runtime > Change runtime type > T4 GPU)")
    else:
        print("nsys still not available after install attempt. "
              "Try: !apt-get install -y nsight-systems-cli")
else:
    print(f"nsys: {shutil.which('nsys')}   GPU: {gpu_name}")
    script = '''
import openimpala as oi, numpy as np, porespy as ps
np.random.seed(42)
data = ps.generators.blobs(shape=[64]*3, porosity=0.5, blobiness=1.5).astype(np.int32)
with oi.Session():
    r = oi.tortuosity(data, phase=0, direction="z", solver="pcg",
                      max_grid_size=32, verbose=0)
    print(f"tau={r.tortuosity:.4f} converged={r.solver_converged}")
'''
    with open("/tmp/oi_profile.py", "w") as f:
        f.write(script)
    !nsys profile --output=/tmp/oi_profile --force-overwrite=true \
        --trace=cuda,nvtx,osrt --cuda-memory-usage=true --stats=true \
        python3 /tmp/oi_profile.py 2>&1 | tail -60
    print("\nReport saved to /tmp/oi_profile.nsys-rep (download for Nsight GUI).")

In [ ]:
if nsys_available:
    stats = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_kern_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if stats.returncode == 0 and stats.stdout.strip():
        lines = stats.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines) if "Time" in l and "Name" in l), None)
        if hdr is not None:
            df_k = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            if "Total Time (ns)" in df_k.columns:
                df_k["Total Time (ms)"] = df_k["Total Time (ns)"] / 1e6
                df_k = df_k.sort_values("Total Time (ns)", ascending=False).head(10)
                print("Top CUDA kernels by total GPU time:")
                print(df_k[["Name", "Total Time (ms)", "Instances"]].to_string(index=False))
            else:
                print(df_k.head().to_string(index=False))
        else:
            print(stats.stdout[:2000])
    else:
        print("nsys stats produced no kernel data.")


In [ ]:
if nsys_available:
    mem = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_mem_size_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if mem.returncode == 0 and mem.stdout.strip():
        lines = mem.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines)
                    if "Operation" in l or "Total" in l), None)
        if hdr is not None:
            df_mem = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            print("CUDA memory transfer summary (H2D/D2H):")
            print(df_mem.to_string(index=False))
            print("\nRed flags:")
            print("  • Frequent H2D transfers of similar size → numpy→VoxelImage")
            print("    copy leaking out of bindings into the hot path")
            print("  • Large D2H transfer per call → result readback overhead")
        else:
            print(mem.stdout[:1500])
    else:
        print("nsys mem report produced no data.")


## 7. Appendix — binding overhead diagnostics

The cells in this section were the original motivation for the notebook: *"is the pybind11 layer slow?"*. Three independent measurements all answer **no**:

* §7a — stage-by-stage timing of a single solve: `solver.value()`   accounts for >99% of wall time at 64³, every other stage is <1%.
* §7b — amortising the VoxelImage across repeated solves:   the speedup is ~1.02×, i.e. nothing to amortise.
* §7c — `cProfile` of the Python call: the binding helpers (parsing,   dispatch) cost <3 ms out of ~26 s.

Keep this in mind for future suspicion that the Python facade is wasteful — it isn't. The bottleneck is the linear solve, which §3 and §4 show MLMG handles optimally.

### 7a. Stage breakdown

`oi.tortuosity(numpy_array, ...)` is not a single operation. From
`facade.py` it is roughly:

```
numpy → VoxelImage.from_numpy    (copy + AMReX BoxArray/Geometry/iMultiFab build)
PercolationCheck(img, ...)        (flood fill — runs every call!)
VolumeFraction(img, ...)          (cheap count)
TortuosityHypre(img, ..., solver) (HYPRE grid/stencil/matrix assembly)
solver.value()                    (actual linear solve + flux integration)
```

We time each stage independently. Whichever bar dominates *is* the
bottleneck. If `from_numpy` or `TortuosityHypre(...)` construction dominates,
your slowdown is in binding/setup — not the solver.

**Note on solver choice for this section**: we time PCG+SMG below because
it's the most direct apples-to-apples comparison with a CPU solver and
its iteration count is informative. On GPU hardware **MLMG is usually
2–5× faster** for this problem class (matrix-free, no assembly cost,
fully GPU-native kernels). §9 runs the full bake-off including MLMG so
you can see the numbers side-by-side.

In [ ]:
from openimpala import _core as _core
from openimpala.facade import _parse_direction, _parse_solver

def time_stages(data, solver="pcg", direction="z", max_grid_size=32, maxiter=1000):
    """Decompose a single tortuosity call into timed binding stages.

    Returns a dict of stage timings plus metadata. If the solver fails to
    converge or produces an invalid result (e.g. boundary flux conservation
    check tripped at large N), the dict still contains every successful stage
    timing and sets _tau / _iters to NaN — callers can then plot or skip the
    failed point without crashing.
    """
    d = _parse_direction(direction)
    st = _parse_solver(solver)
    out = {}

    t0 = time.perf_counter()
    arr = np.ascontiguousarray(data, dtype=np.int32)
    out["np.ascontiguousarray copy"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    img = _core.VoxelImage.from_numpy(arr, max_grid_size)
    out["VoxelImage.from_numpy (AMReX grid + voxel write)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    pc = _core.PercolationCheck(img, 0, d, 0)
    out["PercolationCheck (flood fill)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    vf_val = _core.VolumeFraction(img, 0, 0).value_vf()
    out["VolumeFraction"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    solver_obj = _core.TortuosityHypre(img, vf_val, 0, d, st, ".", 0.0, 1.0, 0, False)
    out["TortuosityHypre ctor (HYPRE setup + matrix assembly)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    try:
        tau = solver_obj.value()
        out["solver.value() (solve + flux)"] = time.perf_counter() - t0
        out["_tau"] = tau
        out["_iters"] = solver_obj.iterations
    except RuntimeError as e:
        out["solver.value() (solve + flux)"] = time.perf_counter() - t0
        out["_tau"] = float("nan")
        out["_iters"] = solver_obj.iterations
        out["_error"] = str(e)

    out["_total"] = sum(v for k, v in out.items() if not k.startswith("_"))
    return out

# Warm-up — first call pays lazy-init inside AMReX/HYPRE
_ = time_stages(datasets[32])

stages = time_stages(data_medium, solver="pcg")
if np.isnan(stages["_tau"]):
    print(f"⚠ solve did not produce a valid tau ({stages.get('_error', 'unknown')}); "
          f"stage timings still shown.\n")
else:
    print(f"tau = {stages['_tau']:.4f}   iters = {stages['_iters']}   total = {stages['_total']:.3f}s\n")
for k, v in stages.items():
    if k.startswith("_"):
        continue
    print(f"  {k:55s}  {v:7.3f}s  ({100*v/stages['_total']:5.1f}%)")

In [ ]:
stage_names = [k for k in stages if not k.startswith("_")]
stage_times = [stages[k] for k in stage_names]

fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
colors = ["#9b59b6", "#8e44ad", "#2ecc71", "#95a5a6", "#f39c12", "#e74c3c"]
ax.barh(y, stage_times, color=colors[:len(stage_names)], alpha=0.85, edgecolor="white")
for i, t in enumerate(stage_times):
    ax.text(t + max(stage_times) * 0.01, i,
            f"{t:.3f}s  ({100*t/stages['_total']:.1f}%)", va="center", fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title("Single oi.tortuosity(64³) call — stage breakdown", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

slowest = max(stage_names, key=lambda k: stages[k])
print(f"\nBottleneck at 64³: {slowest} ({100*stages[slowest]/stages['_total']:.1f}%)")


### 7b. Per-stage scaling

Run the same decomposition at 32³ and 128³ and plot the ratio per stage.
A stage with ratio ≈ `(128/32)³ = 64×` is doing O(N) work — legitimate
compute. A stage with ratio ≈ 1× is constant overhead — every call pays
the same price regardless of problem size, which is the *classic* signature
of binding / setup cost.


In [ ]:
stages_small = time_stages(datasets[32])
stages_big   = time_stages(datasets[128])

if np.isnan(stages_big.get("_tau", 1.0)):
    print(f"Note: 128³ solve did not return a valid tau "
          f"({stages_big.get('_error', 'unknown')}); stage timings still usable "
          f"for the scaling-ratio analysis below.\n")

# Ideal O(N) ratio when scaling voxel count by (128/32)^3
N_ratio = (128 / 32) ** 3
print(f"Ideal O(N) work ratio for 32->128: {N_ratio:.0f}×")
print(f"{'stage':<55s} {'32³ (s)':>10s} {'128³ (s)':>10s} {'ratio':>8s}  behaviour")
print("-" * 110)

rows = []
for k in stage_names:
    ts, tb = stages_small[k], stages_big[k]
    r = tb / max(ts, 1e-9)
    if r > 0.5 * N_ratio:
        label = "scales O(N) ✓"
    elif r < 2.0:
        label = "~constant (overhead)"
    else:
        label = "sub-linear"
    rows.append({"stage": k, "t_32": ts, "t_128": tb, "ratio": r, "label": label})
    print(f"{k:<55s} {ts:>10.3f} {tb:>10.3f} {r:>8.1f}×  {label}")

df_scale = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
w = 0.4
ax.barh(y - w/2, df_scale["t_32"], w, color="#3498db", alpha=0.85, label="32³")
ax.barh(y + w/2, df_scale["t_128"], w, color="#e74c3c", alpha=0.85, label="128³")
for i, r in enumerate(df_scale["ratio"]):
    ax.text(df_scale["t_128"].iloc[i] * 1.02, i + w/2, f"{r:.0f}×",
            va="center", fontsize=8, color="#2c3e50")
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title(f"Per-stage scaling (ideal O(N) ratio = {N_ratio:.0f}×)", fontweight="bold")
ax.axvline(0, color="#bdc3c7", linewidth=0.5)
ax.invert_yaxis()
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

constant_stages = df_scale[df_scale["ratio"] < 2.0]
if not constant_stages.empty:
    print("\n⚠ Stages that DON'T scale with problem size (pure overhead):")
    for _, r in constant_stages.iterrows():
        print(f"    {r['stage']:55s}  ratio={r['ratio']:.1f}×")
    print("  Every oi.tortuosity() call pays these, regardless of image size.")


### 7c. Amortisation test — reusing the VoxelImage

If most per-call time goes into rebuilding AMReX state, then running the solver multiple times against the *same* `VoxelImage` should be markedly cheaper than the equivalent number of calls through the high-level facade. This cell measures the gap.

Two paths are compared on `data_medium`, five calls each:

* **Naive**: five calls to `oi.tortuosity(numpy_array, ...)`. Every call rebuilds the geometry, BoxArray, distribution mapping, iMultiFab, percolation mask, and HYPRE matrix from the bare NumPy input.
* **Amortized**: one call to `_numpy_to_voxelimage` to build the `VoxelImage` once, then five direct calls to `_core.TortuosityHypre` reusing it.

The difference is the per-call setup tax. Sweeping (e.g. directional tortuosity, parameter studies) should always use the amortized pattern.

In [ ]:
from openimpala.facade import _numpy_to_voxelimage, _parse_direction, _parse_solver

def naive_repeat(data, n=5, solver="pcg"):
    """Top-level facade — rebuilds everything every call."""
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        oi.tortuosity(data, phase=0, direction="z", solver=solver,
                      max_grid_size=32, verbose=0)
        ts.append(time.perf_counter() - t0)
    return ts

def amortized_repeat(data, n=5, solver="pcg"):
    """Build VoxelImage once, reuse across solves."""
    d = _parse_direction("z")
    st = _parse_solver(solver)
    img = _numpy_to_voxelimage(data, 32)
    vf = _core.VolumeFraction(img, 0, 0).value_vf()

    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        s = _core.TortuosityHypre(img, vf, 0, d, st, ".", 0.0, 1.0, 0, False)
        s.value()
        ts.append(time.perf_counter() - t0)
    return ts

n_repeat = 5
naive = naive_repeat(data_medium, n_repeat)
amort = amortized_repeat(data_medium, n_repeat)

print(f"Naive  oi.tortuosity(numpy) × {n_repeat}: " + ", ".join(f"{t:.2f}s" for t in naive))
print(f"Amortized (reuse VoxelImage): " + ", ".join(f"{t:.2f}s" for t in amort))
print(f"\nNaive total:      {sum(naive):.2f}s  (mean {np.mean(naive):.2f}s/call)")
print(f"Amortized total:  {sum(amort):.2f}s  (mean {np.mean(amort):.2f}s/call)")
print(f"Speedup from reuse: {sum(naive)/max(sum(amort), 1e-9):.2f}×")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(1, n_repeat + 1)
ax.plot(x, naive, "o-", color="#e74c3c", linewidth=2, markersize=9,
        label="Naive: oi.tortuosity(numpy, ...)")
ax.plot(x, amort, "o-", color="#2ecc71", linewidth=2, markersize=9,
        label="Amortized: reuse VoxelImage + TortuosityHypre")
ax.set_xlabel("Call #")
ax.set_ylabel("Wall time (s)")
ax.set_title("Per-call cost: rebuild-everything vs reuse", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

gap = np.mean(naive) - np.mean(amort)
print(f"Average per-call rebuild tax: {gap:.2f}s "
      f"({100*gap/np.mean(naive):.0f}% of a naive call)")


### 7d. cProfile — Python-level hot frames

cProfile captures cumulative time per Python frame, including time spent inside pybind11 wrappers around C++ calls. This is a different view from §3 — instead of timing named stages, it counts every function call and surfaces unexpected frame costs.

Things to look for in the top 20 frames:

* `_numpy_to_voxelimage`, `VoxelImage.from_numpy` — data ingestion
* `PercolationCheck.__init__` — flood-fill setup
* `TortuosityHypre.__init__` — HYPRE matrix assembly
* `TortuosityHypre.value` — the linear solve

If pure-Python helpers like `_parse_direction` or `_parse_solver` show up high, the binding is doing too much Python-side dispatch.

In [ ]:
pr = cProfile.Profile()
pr.enable()
oi.tortuosity(data_medium, phase=0, direction="z", solver="pcg",
              max_grid_size=32, verbose=0)
pr.disable()

stream = io.StringIO()
ps_stats = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
ps_stats.print_stats(20)
print(stream.getvalue())


## 8. Summary & HPC recommendations


In [ ]:
# Headline numbers from the diagnostics above.
# The fixed-overhead extraction (§4 in pre-v4.3 layouts) was dropped from
# the notebook because three independent measurements (§7a/b/c) all show
# binding overhead is below 1% — there is nothing to extract. So this
# summary now reads only from the appendix stage diagnostic and the
# bake-off / scaling sections.
stage_total = stages["_total"]
stage_top = max((k for k in stages if not k.startswith("_")), key=lambda k: stages[k])
naive_mean = float(np.mean(naive))
amort_mean = float(np.mean(amort))

# Scaling exponent: use the bake-off scaling fit if available, else fall
# back to the per-stage 32->128 ratio computed in §7b.
mlmg_p = fit_results.get("mlmg", float("nan"))
pcg_p = fit_results.get("pcg", float("nan"))
fastest_solver = best_solver
solve_ratio = df_scale.iloc[-1]["ratio"] if "ratio" in df_scale.columns else float("nan")

print("=" * 64)
print("  DIAGNOSIS")
print("=" * 64)
print(f"  Build backend:                  {backend.upper()}"
      + (f"   (host GPU: {gpu_name})" if gpu_name else ""))
print(f"  Single-call bottleneck:         {stage_top}")
print(f"                                  ({100*stages[stage_top]/stage_total:.1f}% of {stage_total:.2f}s)")
if not np.isnan(mlmg_p):
    print(f"  MLMG scaling exponent:          O(N^{mlmg_p:.2f})"
          + ("   (super-linear)" if mlmg_p > 1.2 else "   (near-linear ✓)"))
if not np.isnan(pcg_p):
    print(f"  PCG scaling exponent:           O(N^{pcg_p:.2f})"
          + ("   (super-linear)" if pcg_p > 1.2 else ""))
print(f"  Per-call rebuild tax:           {naive_mean - amort_mean:.2f}s "
      f"({100*(naive_mean-amort_mean)/max(naive_mean,1e-9):.0f}% of naive call)")
print(f"  Fastest solver on this run:     {fastest_solver}")
print(f"  Best max_grid_size:             {best_gs}")
print("=" * 64)

# Headline recommendation — the single most impactful change for this run
print("\n  TOP RECOMMENDATION")
print("  " + "-" * 62)
if gpu_name and not is_gpu_build:
    print(f"  A {gpu_name} is available but the installed wheel is CPU-only.")
    print("  Switch to openimpala-cuda for an expected 10–50× speedup on")
    print("  single-phase 3D diffusion at 128³+. Everything below is")
    print("  secondary until this is resolved.")
elif not np.isnan(mlmg_p) and mlmg_p < 1.2:
    print(f"  MLMG scales near-linearly (p={mlmg_p:.2f}) and is the library")
    print( "  default since v4.3.0. Use solver='auto' (or 'mlmg' explicitly).")
    print( "  HYPRE Krylov methods are available for diagnostic comparison")
    print( "  but scale super-linearly with iteration count on porous media.")
elif not np.isnan(pcg_p) and pcg_p > 1.2:
    print(f"  Compute scales as O(N^{pcg_p:.2f}). Plain Krylov iterations grow")
    print("  with problem size — switch to a multigrid preconditioner")
    print("  (PFMG/SMG via HYPRE, or AMReX MLMG) to restore O(N).")
else:
    print(f"  Solver dominates wall time. Best configuration on this run:")
    print(f"  solver={fastest_solver}, max_grid_size={best_gs}.")
    print("  Further speedup needs algorithmic changes or larger hardware.")

# Amortization recommendation — only fires when the rebuild tax is meaningful
if naive_mean > 2 * amort_mean:
    print()
    print("  Per-call setup is a significant fraction of wall time. For sweeps,")
    print("  build the VoxelImage once and reuse it:")
    print()
    print("    from openimpala.facade import _numpy_to_voxelimage")
    print("    from openimpala import _core")
    print("    img = _numpy_to_voxelimage(arr, max_grid_size)")
    print("    s   = _core.TortuosityHypre(img, vf, 0, d, st, '.', 0, 1, 0, False)")
    print("    s.value()")
    print()
    print("  …rather than calling oi.tortuosity(numpy_array, ...) per iteration.")

In [ ]:
inputs_template = f"""# Recommended OpenImpala inputs (generated by profiling notebook)
filename = your_image.tif
phase_id = 0
direction = ALL
solver_type = {best_solver}
box_size = {best_gs}
calculation_method = flow_through
hypre.maxiter = 1000
hypre.eps = 1e-9
physics.type = diffusion
verbose = 1
"""
print(inputs_template)


## Cleanup


In [ ]:
try:
    from openimpala._core import amrex_initialized
    if amrex_initialized():
        session.__exit__(None, None, None)
        print("Session closed.")
    else:
        print("Session already finalized.")
except Exception:
    print("Cleanup done.")
